# Smart-Demand: Category-Level Sales Prediction (Leak-Free)

**Machine Learning | Final Project**

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Task:** Regression: predict next-month order quantity per category  
**Main Model:** Random Forest Regressor  
**Comparison Model:** Linear Regression  
**Baselines:** Last month and the previous three-month average

This notebook develops one-month-ahead forecasts for each product category, from selected Olist CSV columns through integration, historical features, chronological evaluation, and model export. It follows the structure of notebook 02 while addressing the identified sources of leakage.

The target includes all recorded order items by purchase month, including later cancellations. It differs from notebook 02's delivered-order target, so scores are not directly comparable.

Forecasts assume item prices and freight were known at purchase. The static archive does not track historical revisions.

## 1. Setup & Library Import

Import the libraries used for data preparation, visualization, regression, and saving model artifacts. Set a fixed random seed for reproducibility and define the test fraction used later in the chronological split. Both models use the same prepared feature set.

### 1.1 Library Import

In [ ]:
import os
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.2

plt.style.use("seaborn-v0_8-whitegrid")
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Libraries loaded.")

## 2. Load Dataset

Load the required columns from five Olist CSVs: orders, items, products, reviews, and category translation. Selecting columns before joining keeps the input tables focused on the information needed for category-level forecasting. The source folder is the same as notebook 02.

Review answer timestamps are included to check when ratings became available. Payments are excluded because payment–price differences have not been verified as actual discounts.

### 2.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = '/content/drive/MyDrive/dataset/'
MODEL_PATH = '/content/drive/MyDrive/SmartDemand_Dataset/models/'
os.makedirs(MODEL_PATH, exist_ok=True)
print('Drive mounted.')

### 2.2 Load Selected Columns

In [ ]:
df_orders = pd.read_csv(
    DATA_PATH + "olist_orders_dataset.csv",
    usecols=["order_id", "order_purchase_timestamp"],
)
df_items = pd.read_csv(
    DATA_PATH + "olist_order_items_dataset.csv",
    usecols=["order_id", "order_item_id", "product_id", "price", "freight_value"],
)
df_products = pd.read_csv(
    DATA_PATH + "olist_products_dataset.csv",
    usecols=["product_id", "product_category_name"],
)
df_reviews = pd.read_csv(
    DATA_PATH + "olist_order_reviews_dataset.csv",
    usecols=["order_id", "review_score", "review_answer_timestamp"],
)
df_cat_transl = pd.read_csv(
    DATA_PATH + "product_category_name_translation.csv",
    usecols=["product_category_name", "product_category_name_english"],
)

for name, df in [
    ("orders", df_orders), ("items", df_items),
    ("products", df_products), ("reviews", df_reviews), ("translation", df_cat_transl),
]:
    print(f"{name:12s}: {df.shape}")

## 3. Data Integration

Combine order items with purchase dates, product categories, and available order ratings. Validate table keys and many-to-one joins so each master row still represents one order item. Aggregate the result into one row per category and purchase month.

Average reviews per order only when answered before the end of its purchase month. Late or undated reviews are excluded from that month's rating feature.

Infer the date range from the CSVs and omit the latest month as potentially incomplete. Coverage of earlier months must also be checked against the source archive.

### 3.1 Merge Source Tables

In [ ]:
for name, df, keys in [
    ("orders", df_orders, ["order_id"]),
    ("items", df_items, ["order_id", "order_item_id"]),
    ("products", df_products, ["product_id"]),
    ("translation", df_cat_transl, ["product_category_name"]),
]:
    if df[keys].isna().any().any() or df.duplicated(keys).any():
        raise ValueError(f"{name}: ada kunci kosong atau duplikat pada {keys}")

df_orders["order_purchase_timestamp"] = pd.to_datetime(
    df_orders["order_purchase_timestamp"], errors="coerce"
)
if df_orders["order_purchase_timestamp"].isna().any():
    raise ValueError("Ada tanggal pembelian kosong/tidak valid.")

df_reviews["review_answer_timestamp"] = pd.to_datetime(
    df_reviews["review_answer_timestamp"], errors="coerce"
)
df_reviews["review_score"] = pd.to_numeric(df_reviews["review_score"], errors="coerce")

# Cutoff per cohort bulan pembelian: hanya skor yang sudah diketahui pada akhir bulan.
review_history = df_reviews.merge(
    df_orders, on="order_id", how="left", validate="many_to_one"
)
review_cutoff = (
    review_history["order_purchase_timestamp"].dt.to_period("M") + 1
).dt.to_timestamp()
available_review = (
    review_history["review_score"].between(1, 5)
    & review_history["review_answer_timestamp"].notna()
    & (review_history["review_answer_timestamp"] >= review_history["order_purchase_timestamp"])
    & (review_history["review_answer_timestamp"] < review_cutoff)
)
review_per_order = (
    review_history.loc[available_review]
    .groupby("order_id", as_index=False)["review_score"].mean()
    .rename(columns={"review_score": "rating_available"})
)

master = df_items.merge(
    df_orders, on="order_id", how="left", validate="many_to_one"
)
if master["order_purchase_timestamp"].isna().any():
    raise ValueError("Ada item tanpa pasangan order.")
master = master.merge(
    df_products, on="product_id", how="left", validate="many_to_one"
)
master = master.merge(
    df_cat_transl, on="product_category_name", how="left", validate="many_to_one"
)
master = master.merge(
    review_per_order, on="order_id", how="left", validate="many_to_one"
)
master["category"] = (
    master["product_category_name_english"]
    .fillna(master["product_category_name"])
    .fillna("unknown")
)
master["period"] = master["order_purchase_timestamp"].dt.to_period("M")
assert len(master) == len(df_items), "Join menggandakan item."

for col in ["price", "freight_value"]:
    master[col] = pd.to_numeric(master[col], errors="coerce")
    invalid = ~np.isfinite(master[col]) | (master[col] < 0)
    master.loc[invalid, col] = np.nan

excluded_month = master["period"].max()
master_history = master[master["period"] < excluded_month].copy()
if master_history.empty:
    raise ValueError("Histori belum cukup setelah bulan terakhir dikeluarkan.")

print("Review rows available before their purchase-month cutoff:", int(available_review.sum()))
print("Master rows:", len(master))
print("Bulan terakhir CSV yang dikecualikan:", excluded_month)
display(master_history.head())

### 3.2 Aggregate to Category-Month Level

In [ ]:
df_agg = (
    master_history.groupby(["category", "period"])
    .agg(
        quantity_ordered=("order_item_id", "size"),
        avg_price=("price", "mean"),
        avg_freight=("freight_value", "mean"),
        avg_rating=("rating_available", "mean"),
        num_products=("product_id", "nunique"),
        min_price=("price", "min"),
        max_price=("price", "max"),
    )
    .reset_index()
)
df_agg["price_range"] = df_agg["max_price"] - df_agg["min_price"]
df_agg = df_agg.drop(columns=["min_price", "max_price"])

print("Category-month rows:", len(df_agg))
print("Categories:", df_agg["category"].nunique())
display(df_agg.head())

## 4. Data Cleaning

Build a continuous monthly calendar for each category from its first purchase month. If the archive is complete, a month without orders has zero ordered units and products, while price, freight, and rating remain missing. This keeps calendar gaps from being mistaken for adjacent months.

Retain valid sales spikes instead of removing rows based on the full target distribution. Missing numeric inputs are imputed later using training data only, and categories need three months of history before entering evaluation.

### 4.1 Missing Values and Monthly Calendar

In [ ]:
calendar = []
for category, first_month in df_agg.groupby("category")["period"].min().items():
    calendar.append(pd.DataFrame({
        "category": category,
        "period": pd.period_range(first_month, excluded_month - 1, freq="M"),
    }))

df_clean = pd.concat(calendar, ignore_index=True).merge(
    df_agg, on=["category", "period"], how="left", validate="one_to_one"
)
for col in ["quantity_ordered", "num_products"]:
    df_clean[col] = df_clean[col].fillna(0).astype(int)

df_clean = df_clean.sort_values(["category", "period"]).reset_index(drop=True)
assert not df_clean.duplicated(["category", "period"]).any()
assert df_clean["quantity_ordered"].sum() == len(master_history)

print("Monthly rows after cleaning:", len(df_clean))
print("History:", df_clean["period"].min(), "to", df_clean["period"].max())
print("Missing numeric values, including rating, will be imputed from training data.")

## 5. Feature Engineering

Create predictors that would be available before the forecast month starts. Sales use a shifted three-month average, while transaction conditions and time-filtered ratings come from the previous month. The seasonality lookup is calculated only after the time split.

Features used in this model:

- sales_avg_3m - average category orders over the previous three calendar months
- prev_avg_price - average item price in the previous month
- prev_avg_freight - average freight cost in the previous month
- prev_avg_rating - previous-month order rating available before the forecast month
- prev_num_products - number of unique products ordered in the previous month
- prev_price_range - difference between the highest and lowest item prices in the previous month
- category - product category, encoded inside the model pipeline
- seasonality_index - global monthly demand index calculated from training data only

Last month's sales are used only as a baseline, not as a model feature.

In [ ]:
def build_features(monthly_data):
    # Fungsi yang sama dipakai untuk training dan prediksi bulan berikutnya.
    df = monthly_data.sort_values(["category", "period"]).reset_index(drop=True).copy()
    for category, rows in df.groupby("category"):
        if rows["period"].duplicated().any():
            raise ValueError(f"Duplikasi bulan pada kategori {category}")
        if len(rows) > 1 and not (np.diff(rows["period"].array.asi8) == 1).all():
            raise ValueError(f"Kalender kategori {category} tidak berurutan.")

    group = df.groupby("category")
    df["history_months"] = group.cumcount()
    df["sales_avg_3m"] = group["quantity_ordered"].transform(
        lambda sales: sales.shift(1).rolling(3, min_periods=3).mean()
    )
    df["last_month_sales"] = group["quantity_ordered"].shift(1)
    for col in ["avg_price", "avg_freight", "avg_rating", "num_products", "price_range"]:
        df["prev_" + col] = group[col].shift(1)
    return df[df["history_months"] >= 3].copy()

df_feat = build_features(df_clean)

FEATURES = [
    "sales_avg_3m",
    "prev_avg_price",
    "prev_avg_freight",
    "prev_avg_rating",
    "prev_num_products",
    "prev_price_range",
    "category",
    "seasonality_index",
]
TARGET = "quantity_ordered"
NUMERIC_FEATURES = [col for col in FEATURES if col != "category"]

assert df_feat["sales_avg_3m"].notna().all()
assert TARGET not in FEATURES and "last_month_sales" not in FEATURES
print("Features:", FEATURES)
print("Target:", TARGET)
print("Rows:", len(df_feat), "| Model features:", len(FEATURES))
print("seasonality_index will be calculated from training after the time split.")
display(df_feat[["category", "period", "sales_avg_3m", TARGET]].head())

## 6. Exploratory Data Analysis (EDA)

Examine the training target distribution, numeric correlations, leading categories, monthly trends, and the relationship between historical and current sales. These plots help explain differences in category scale and highlight patterns or data-quality issues.

Only training months are used for EDA. Test observations remain outside feature and parameter decisions.

### 6.1 Target Distribution

In [ ]:
months = pd.PeriodIndex(sorted(df_feat["period"].unique()), freq="M")
split_index = int(len(months) * (1 - TEST_SIZE))
if split_index < 6 or split_index >= len(months):
    raise ValueError("Butuh minimal enam bulan training dan satu bulan test.")
if not (np.diff(months.asi8) == 1).all():
    raise ValueError("Ada bulan tanpa data evaluable; periksa histori sumber.")

df_eda = df_feat[df_feat["period"].isin(months[:split_index])].copy()
eda_features = [col for col in NUMERIC_FEATURES if col in df_eda.columns]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df_eda[TARGET], bins=40, color="steelblue", edgecolor="white")
axes[0].set(title="Training Target Distribution",
            xlabel="Units ordered", ylabel="Frequency")
axes[1].hist(np.log1p(df_eda[TARGET]), bins=40, color="cornflowerblue", edgecolor="white")
axes[1].set(title="Log-Scale Training Target",
            xlabel="log(Units ordered + 1)", ylabel="Frequency")
plt.tight_layout()
plt.show()
display(df_eda[eda_features + [TARGET]].describe().round(2))

### 6.2 Correlation Heatmap

In [ ]:
corr_matrix = df_eda[eda_features + [TARGET]].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
values = np.ma.masked_where(mask, corr_matrix.to_numpy())
heatmap = ax.imshow(values, cmap="Blues", vmin=-1, vmax=1)
labels = corr_matrix.columns
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for row in range(len(labels)):
    for col in range(row):
        value = corr_matrix.iloc[row, col]
        if pd.notna(value):
            ax.text(col, row, f"{value:.2f}", ha="center", va="center")
fig.colorbar(heatmap, ax=ax, label="Correlation")
ax.set_title("Feature Correlations — Training Only")
plt.tight_layout()
plt.show()

### 6.3 Top 15 Categories by Total Sales

In [ ]:
top_categories = df_eda.groupby("category")[TARGET].sum().nlargest(15)
top_categories.sort_values().plot.barh(
    figsize=(11, 5), color="steelblue"
)
plt.title("Top 15 Categories — Training Only")
plt.xlabel("Total units ordered")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 6.4 Monthly Sales Trend

In [ ]:
training_trend = df_eda.groupby("period")[TARGET].sum()

plt.figure(figsize=(12, 4))
plt.plot(training_trend.index.to_timestamp(), training_trend.values, marker="o")
plt.title("Monthly Sales Trend — Training Only")
plt.xlabel("Month")
plt.ylabel("Units ordered")
plt.tight_layout()
plt.show()

### 6.5 Last Month Sales vs Current Sales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feature, title in [
    (axes[0], "last_month_sales", "Last Month vs Current Sales"),
    (axes[1], "sales_avg_3m", "Previous 3-Month Average vs Current Sales"),
    (axes[2], "prev_num_products", "Previous Product Count vs Current Sales"),
]:
    ax.scatter(df_eda[feature], df_eda[TARGET], alpha=0.4, s=15)
    ax.set(title=title, xlabel=feature, ylabel="Current units ordered")
plt.tight_layout()
plt.show()

## 7. Train-Test Split

Use the earliest 80% of months for training and the latest 20% for testing. Keep all categories from a given month in the same split, so the evaluation represents forecasting later periods from earlier observations. There is no random split, validation set, or tuning.

Calculate global seasonality from training targets and apply the same lookup to both splits. Unseen calendar months use the mean training index.

During one-month-ahead testing, model parameters stay fixed while observed history updates after each month ends. This is not a multi-month forecast made from one fixed origin.

### 7.1 Chronological Split and Training Seasonality

In [ ]:
test_start = months[split_index]

train = df_feat[df_feat["period"] < test_start].copy()
test = df_feat[df_feat["period"] >= test_start].copy()
assert train["period"].max() < test["period"].min()

# Sama seperti pendekatan teman: mean target per bulan kalender, training saja.
monthly_avg = train.groupby(train["period"].dt.month)[TARGET].mean()
season_range = monthly_avg.max() - monthly_avg.min()
if season_range > 0:
    monthly_norm = (monthly_avg - monthly_avg.min()) / season_range
else:
    monthly_norm = pd.Series(0.5, index=monthly_avg.index)

for part in [train, test]:
    part["seasonality_index"] = (
        part["period"].dt.month.map(monthly_norm).fillna(monthly_norm.mean())
    )

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

split_summary = pd.DataFrame([
    {"Split": name, "Start": str(df["period"].min()), "End": str(df["period"].max()),
     "Months": df["period"].nunique(), "Rows": len(df)}
    for name, df in [("Train", train), ("Test", test)]
])
display(split_summary)
print(f"Train: {len(train):,} rows | Test: {len(test):,} rows")

### 7.2 Training-Only Preprocessing

In [ ]:
# Imputer, scaler, dan encoder hanya di-fit pada X_train.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median", add_indicator=True, keep_empty_features=True
    )),
    ("scaler", StandardScaler()),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["category"]),
])

def evaluate_predictions(actual, predicted):
    predicted = np.maximum(np.asarray(predicted, dtype=float), 0)
    actual = np.asarray(actual, dtype=float)
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": float(r2_score(actual, predicted)) if np.ptp(actual) > 0 else None,
    }

## 8. Model Training

Train Linear Regression as a comparison model and Random Forest as the predefined main model. Both receive the same features. Linear Regression provides a linear reference, while RF can capture nonlinear relationships and interactions.

Keep RF settings identical to notebook 02: 300 trees, depth 20, minimum split 5, minimum leaf 2, and sqrt features. Fit all preprocessing on training data only. Clip negative predictions to zero and evaluate without rounding.

### 8.1 Linear Regression

In [ ]:
print("Training Linear Regression...")
lr_model = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", LinearRegression()),
])
lr_model.fit(X_train, y_train)
print("Done.")

### 8.2 Random Forest Regressor

In [ ]:
print("Training Random Forest Regressor...")
rf_model = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
rf_model.fit(X_train, y_train)

# Model utama ditetapkan sebelum hasil test dilihat.
selected_model_name = "Random Forest"
print("Done. Last training month:", train["period"].max())

## 9. Evaluation & Comparison

Evaluate RF, Linear Regression, and both simple baselines on the same test rows. The baselines predict either last month's quantity or the previous three-month average, showing whether ML improves on straightforward use of sales history.

MAE measures average absolute error in units; RMSE gives more weight to large errors. R² compares squared error with predicting the overall test mean and is not percentage accuracy.

RF remains the predefined main model. Report any baseline underperformance without retuning or choosing a replacement from these test results.

Display the four methods in one `test_result` table with MAE, RMSE, and R². Predictions stay in memory for evaluation and plots; no row-level prediction report is generated.

### 9.1 Test Result

In [ ]:
lr_pred = np.maximum(lr_model.predict(X_test), 0)
rf_pred = np.maximum(rf_model.predict(X_test), 0)
test_prediction_map = {
    "Last Month Baseline": test["last_month_sales"].to_numpy(),
    "3-Month Average Baseline": test["sales_avg_3m"].to_numpy(),
    "Linear Regression": lr_pred,
    "Random Forest": rf_pred,
}
test_result = pd.DataFrame([
    {"Model": name, **evaluate_predictions(y_test, pred)}
    for name, pred in test_prediction_map.items()
]).set_index("Model")

display(test_result.round(4))

### 9.2 Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, name in zip(
    axes, [lr_pred, rf_pred], ["Linear Regression", "Random Forest"]
):
    ax.scatter(y_test, pred, alpha=0.4, s=15)
    limit = max(1, float(max(y_test.max(), np.max(pred))))
    ax.plot([0, limit], [0, limit], "k--")
    ax.set(title=f"{name}: test R2 = {r2_score(y_test, pred):.4f}",
           xlabel="Actual units ordered", ylabel="Predicted units")
plt.tight_layout()
plt.show()

### 9.3 Residual Analysis

In [ ]:
selected_pred = np.asarray(test_prediction_map[selected_model_name])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
residuals = y_test.to_numpy() - selected_pred
axes[0].scatter(selected_pred, residuals, alpha=0.4, s=15)
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set(title="Selected Model — Test Residuals", xlabel="Predicted", ylabel="Actual - predicted")
axes[1].hist(residuals, bins=30, color="steelblue", edgecolor="white")
axes[1].set(title="Residual Distribution", xlabel="Residual", ylabel="Frequency")
plt.tight_layout()
plt.show()

## 10. Feature Importance

Inspect how the training-fitted RF distributes importance across numeric inputs and encoded category features. The table lists the leading inputs, and the chart displays the top 15 for readability.

Importance describes the fitted model, not a causal effect. Interpret it alongside test errors and baseline comparisons.

### 10.1 Random Forest Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    "feature": rf_model.named_steps["preprocess"].get_feature_names_out(),
    "importance": rf_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)

display(importance_df.head(20))
importance_df.head(15).sort_values("importance").plot.barh(
    x="feature", y="importance", figsize=(10, 6), color="steelblue", legend=False
)
plt.title("Random Forest Feature Importance — Top 15")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 11. Save Model

Export the fitted models and category encoder so prediction can be reproduced without retraining. Save three separate artifacts in `models/`, the same output folder as notebook 02:

- `random_forest_model.joblib`: fitted RF pipeline.
- `linear_regression_model.joblib`: fitted Linear Regression pipeline.
- `label_encoder.joblib`: the fitted category encoder exported from the RF pipeline.

The encoder is a **OneHotEncoder**, despite the legacy filename. Both pipelines already include preprocessing; do not encode categories again before prediction. Seasonality, feature order, encoder type, and test metrics are stored in `config.json`.

The evaluation summary appears only as the `test_result` cell table. No CSV files or additional forecast reports are exported. Reload the saved artifacts to verify that predictions remain unchanged.

Saving replaces matching model files and `config.json`; files from older runs are not deleted. No `final_model.joblib` is generated.

### 11.1 Save Models and Configuration

In [ ]:
# Export the fitted encoder already used by the RF; do not fit a new encoder.
label_enc = rf_model.named_steps["preprocess"].named_transformers_["category"]

joblib.dump(rf_model, MODEL_PATH + "random_forest_model.joblib")
joblib.dump(lr_model, MODEL_PATH + "linear_regression_model.joblib")
joblib.dump(label_enc, MODEL_PATH + "label_encoder.joblib")

config = {
    "target": TARGET,
    "target_definition": "count_order_items_by_purchase_month_all_recorded_orders",
    "features": FEATURES,
    "history_window_months": 3,
    "seasonality_index": {int(month): float(value) for month, value in monthly_norm.items()},
    "seasonality_fallback": float(monthly_norm.mean()),
    "category_encoder_file": "label_encoder.joblib",
    "category_encoder_type": type(label_enc).__name__,
    "category_encoder_included_in_model_pipelines": True,
    "categories": [str(category) for category in label_enc.categories_[0]],
    "rating_policy": "purchase_month_reviews_answered_before_next_month_start",
    "review_availability_timestamp": "review_answer_timestamp",
    "history_start": str(df_clean["period"].min()),
    "history_end": str(df_clean["period"].max()),
    "excluded_latest_source_month": str(excluded_month),
    "forecast_protocol": "one_month_ahead_with_updated_observed_history",
    "cold_start_categories_evaluated": False,
    "selected_model": selected_model_name,
    "model_selection_policy": "random_forest_fixed_before_test_no_tuning",
    "test_fraction_of_calendar_months": TEST_SIZE,
    "rf_parameters": {
        key: rf_model.named_steps["model"].get_params()[key]
        for key in [
            "n_estimators", "max_depth", "min_samples_split", "min_samples_leaf",
            "max_features", "bootstrap", "random_state", "n_jobs",
        ]
    },
    "trained_through": str(train["period"].max()),
    "split_summary": split_summary.to_dict(orient="records"),
    "test_metrics": json.loads(test_result.to_json(orient="index")),
    "prediction_postprocessing": "clip_lower_zero_no_rounding",
    "source_time_assumption": "items, prices and freight available at purchase; static archive",
    "random_state": RANDOM_STATE,
    "versions": {
        "numpy": np.__version__, "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__, "joblib": joblib.__version__,
    },
}
with open(MODEL_PATH + "config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2, ensure_ascii=False, allow_nan=False)

print("Saved to:", MODEL_PATH)
print("Main model:", selected_model_name)

### 11.2 Load and Verify the Saved Model

In [ ]:
def predict_saved_model(pipeline, rows, saved_config):
    predictions = pipeline.predict(rows[saved_config["features"]])
    return np.maximum(np.asarray(predictions, dtype=float), 0)

loaded_rf = joblib.load(MODEL_PATH + "random_forest_model.joblib")
loaded_lr = joblib.load(MODEL_PATH + "linear_regression_model.joblib")
loaded_encoder = joblib.load(MODEL_PATH + "label_encoder.joblib")
with open(MODEL_PATH + "config.json", encoding="utf-8") as file:
    loaded_config = json.load(file)

np.testing.assert_allclose(
    predict_saved_model(loaded_rf, test, loaded_config), rf_pred,
    rtol=1e-12, atol=1e-12,
)
np.testing.assert_allclose(
    predict_saved_model(loaded_lr, test, loaded_config), lr_pred,
    rtol=1e-12, atol=1e-12,
)
np.testing.assert_array_equal(
    loaded_encoder.transform(test[["category"]]),
    loaded_rf.named_steps["preprocess"].named_transformers_["category"]
    .transform(test[["category"]]),
)
print("Both reloaded models and the exported encoder match the training artifacts.")

## Summary

- Target: order items by purchase month.
- Inputs: prior three-month sales average, previous-month conditions, and time-filtered ratings.
- Seasonality: training-only global monthly index.
- Evaluation: chronological 80/20 split, fixed RF parameters, no tuning.

Report test metrics against both baselines, including any RF underperformance. Results exclude categories with fewer than three months of history and depend on archive completeness and timestamp accuracy.

Update the app's input schema and preprocessing before installing these artifacts.